#Exercícios para entregar 2

###Aluno: Katlyn Ribeiro           
###

In [50]:
#imports necessarios
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KernelDensity
from scipy.stats import multivariate_normal

In [26]:
#serao utilizados as bases pegando direto do drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1 - Faça a classificação da base BreastCancer usando o classificador Naive Bayes.

In [32]:
#caminho para a base BreastCancer pelo drive
file_path= "/content/drive/MyDrive/Aprendizado_de_maquina_exercicios/BreastCancer.csv"
dataframe = pd.read_csv(file_path, na_values='?')
#remove as linhas vazias
dataframe = dataframe.dropna()

#retira a coluna Id se caso ela exista
if 'Id' in dataframe.columns:
    dataframe = dataframe.drop('Id', axis=1)

#variaveis de entrada-----------
X = dataframe.drop('Class', axis=1).values
#variavel alvo---------------
y = dataframe['Class'].values

#aqui acontece a divisão de treino e teste onde foi separado em 40% dos dados para treino e 60% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.6, random_state=42)

#aqui acontece o treinamento do classificador Naive Bayes
modelo = GaussianNB()
modelo.fit(X_treino, y_treino)

#por fim preve os y's para os dados de teste
y_previsao = modelo.predict(X_teste)

#calcula a acuracia
print('Acuracia da base "BreastCancer" com o uso do Classificador Naive Bayes:', accuracy_score(y_teste, y_previsao))

Acuracia da base "BreastCancer" com o uso do Classificador Naive Bayes: 0.9682926829268292


2 - Considere a base vertebralcolumn-3C e compare o classificadores: Naive Bayes, Classificador Bayesiano paramétrico e o classificador Bayesiano não-paramétrico.

In [79]:
#caminho para a base vertebralcolumn-3C pelo drive
file_path= "/content/drive/MyDrive/Aprendizado_de_maquina_exercicios/vertebralcolumn-3C.csv"
data = pd.read_csv(file_path)
#remove as linhas vazias
data = data.dropna()

#essa parte seleciona todas as linhas, todas as colunas (menos a ultima) e transforma em array

#variáveis de entrada---------
X = data.iloc[:, :-1].values
#variavel alvo---------------
y = data.iloc[:, -1].values
#armazena as classes
classes = np.unique(y)

#aqui acontece a divisão de treino e teste onde foi separado em 70% dos dados para treino e 30% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42)

#agora acontece a separação dos 3 modelos que vao ser comparados:

#Primeiro temos o Classificador Naive Bayes
NB_modelo = GaussianNB()
NB_modelo.fit(X_treino, y_treino)
NB_previsao = NB_modelo.predict(X_teste)
print('- Modelo 1: --------------------------------------------------------------------------')
print('Acuracia com o uso do Classificador Naive Bayes:', accuracy_score(y_teste, NB_previsao))

#Segundo o Classificador Bayesiano Parametrico
P_param = pd.DataFrame(data=np.zeros((X_teste.shape[0], len(classes))), columns=classes)
Pc = np.zeros(len(classes))

for i in range(len(classes)):
    elementos = np.where(y_treino == classes[i])[0]
    Pc[i] = len(elementos) / len(y_treino)

    Z = X_treino[elementos, :].astype(float)
    m = np.mean(Z, axis=0)
    cv = np.cov(Z, rowvar=False)

    for j in range(X_teste.shape[0]):
        x = X_teste[j, :]
        pj = multivariate_normal.pdf(x, mean=m, cov=cv, allow_singular=True)
        P_param.loc[j, classes[i]] = pj * Pc[i]

y_param_previsao = []

for i in range(X_teste.shape[0]):
    c = np.argmax(P_param.iloc[i].values)
    y_param_previsao.append(classes[c])

print('- Modelo 2: --------------------------------------------------------------------------')
print('Acuracia com o uso do Classificador Bayesiano Paramétrico:', accuracy_score(y_teste, y_param_previsao))

#Terceiro o Classificador Bayesiano não-paramétrico
h = 1.0
P_nparam = pd.DataFrame(data=np.zeros((X_teste.shape[0], len(classes))), columns=classes)

for i in range(len(classes)):
    elements = np.where(y_treino == classes[i])[0]
    Pc[i] = len(elements) / len(y_treino)
    Z = X_treino[elements, :].astype(float)

    kde = KernelDensity(kernel='gaussian', bandwidth=h).fit(Z)

    for j in range(X_teste.shape[0]):
        x = X_teste[j, :].reshape(1, -1)
        pj = np.exp(kde.score_samples(x))[0]
        P_nparam.loc[j, classes[i]] = pj * Pc[i]

y_previsao_nparam = []

for i in range(X_teste.shape[0]):
    c = np.argmax(P_nparam.iloc[i].values)
    y_previsao_nparam.append(classes[c])

print('- Modelo 3: --------------------------------------------------------------------------')
print('Acuracia com o uso do Classificador Bayesiano Não-Paramétrico:', accuracy_score(y_teste, y_previsao_nparam))
print('--------------------------------------------------------------------------------------')

- Modelo 1: --------------------------------------------------------------------------
Acuracia com o uso do Classificador Naive Bayes: 0.7849462365591398
- Modelo 2: --------------------------------------------------------------------------
Acuracia com o uso do Classificador Bayesiano Paramétrico: 0.8064516129032258
- Modelo 3: --------------------------------------------------------------------------
Acuracia com o uso do Classificador Bayesiano Não-Paramétrico: 0.8279569892473119
--------------------------------------------------------------------------------------


3 - Implemente uma versão do classificador Bayesiano usando orientação ao objeto. Implemente os métodos fit e predict.

In [62]:
#classe do classificador bayesiano com o uso de orientação a objetos
class ClassificadorBayesiano:
    #construtor
    def __init__(self):
        self.classes = None
        self.class_p = {}
        self.class_media = {}
        self.class_covariancia = {}

    #treinamento
    def fit(self, X, y):
        self.classes = np.unique(y)
        #calcula p, a media e a covariancia
        for c in self.classes:
            X_c = X[y == c].astype(float)
            self.class_p[c] = X_c.shape[0] / X.shape[0]
            self.class_media[c] = np.mean(X_c, axis=0)
            self.class_covariancia[c] = np.cov(X_c, rowvar=False)

    #previsao
    def predict(self, X):
        y_previsao = []

        for x in X:
            posterior = {}

            for c in self.classes:
                p_c = self.class_p[c]
                #calcula o posterior de cada classe
                likelihood = multivariate_normal.pdf(x, mean=self.class_media[c], cov=self.class_covariancia[c], allow_singular=True)
                posterior[c] = likelihood * p_c

            y_previsao.append(max(posterior, key=posterior.get))

        return np.array(y_previsao)

#utiliza a classe
clf = ClassificadorBayesiano()
clf.fit(X_treino, y_treino)

y_previsao_customizada = clf.predict(X_teste)

print('--------------------------------------------------------------------------------------')
print('Acuracia com o uso da classe ClassificadorBayesiano criado:', accuracy_score(y_teste, y_previsao_customizada))
print('--------------------------------------------------------------------------------------')

--------------------------------------------------------------------------------------
Acuracia com o uso da classe ClassificadorBayesiano criado: 0.8064516129032258
--------------------------------------------------------------------------------------


4 - Verifique essa propriedade: Apesar da limitação em assumir independência dos atributos, o classificador Naive Bayes é robusto.

Gere dados com diferentes níveis de correlação entre as variáveis e verifique se a perfomance do algoritmo Naive Bayes muda com a correlação.

In [69]:
#funcao para analisar o desempenho do classificador Naive Bayes quando variamos o nivel de correlaçao
def variando_nivel_de_correlacao_naive_bayes(correlacao, n=1000):

    #-----------------------------------------------
    media0 = [0, 0]
    cov0 = [[1, correlacao], [correlacao, 1]]
    X0 = np.random.multivariate_normal(media0, cov0, n // 2)
    y0 = np.zeros(n // 2)

    #----------------------------------------------
    media1 = [2, 2]
    cov1 = [[1, correlacao], [correlacao, 1]]
    X1 = np.random.multivariate_normal(media1, cov1, n // 2)
    y1 = np.ones(n // 2)

    #---------------------------------------------
    X = np.vstack((X0, X1))
    y = np.hstack((y0, y1))

    #aqui acontece a divisão de treino e teste, onde foi separado em 30% dos dados para treino e 70% para teste
    X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.7, random_state=42)

    modelo = GaussianNB()
    modelo.fit(X_treino, y_treino)

    y_previsao = modelo.predict(X_teste)

    return accuracy_score(y_teste, y_previsao)

#sem correlaçao ate totalmente correlacionado
correlacao = [0.0, 0.3, 0.6, 0.9, 0.99]

print('--------------Correlação Vs acuracia com o classficados Naive Bayes-------------------')
for corr in correlacao:
    acc = variando_nivel_de_correlacao_naive_bayes(corr)
    print(f'Com a correlação: {corr:.2f} a acuracia com o classificador NB fica: {acc:.4f}')
print('--------------------------------------------------------------------------------------')

--------------Correlação Vs acuracia com o classficados Naive Bayes-------------------
Com a correlação: 0.00 a acuracia com o classificador NB fica: 0.9171
Com a correlação: 0.30 a acuracia com o classificador NB fica: 0.8743
Com a correlação: 0.60 a acuracia com o classificador NB fica: 0.8571
Com a correlação: 0.90 a acuracia com o classificador NB fica: 0.8514
Com a correlação: 0.99 a acuracia com o classificador NB fica: 0.8443
--------------------------------------------------------------------------------------


5 - Implemente o Naive Bayes sem usar o scik-learn. Use distribuições de probabilidade diferentes da normal.

In [78]:
#classe do classificador Naies Bayes com a exponencial
class Exponencial_NaiveBayes:
    #construtor
    def __init__(self):
        self.classes = None
        self.class_p = {}
        self.class_lambda = {}

    #treinamento
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.class_p = {}
        self.class_lambda = {}

        for c in self.classes:
            X_c = X[y == c].astype(float)

            self.class_p[c] = X_c.shape[0] / X.shape[0]
            medias = np.mean(X_c, axis=0)
            medias[medias == 0] = 1e-6
            self.class_lambda[c] = 1.0 / medias

    #definiçao da distribuiçao exponencial
    def _pdf(self, x, class_lambdaa):
        x_nn_negativo = np.maximum(x, 0)

        return class_lambdaa * np.exp(-class_lambdaa * x_nn_negativo)

    #previsao
    def previsao(self, X):
        y_previsao = []

        for x in X:
            posterior = {}

            for c in self.classes:
                p = np.log(self.class_p[c])
                #evita underflow numerico
                k = np.sum(np.log(self._pdf(x, self.class_lambda[c]) + 1e-10))
                posterior[c] = p + k

            y_previsao.append(max(posterior, key=posterior.get))

        return np.array(y_previsao)

#------------------------------------------------------------------------------------
#irei testar o classificador Naive Bayes de Exponencial com a base BreastCancer
file_path= "/content/drive/MyDrive/Aprendizado_de_maquina_exercicios/BreastCancer.csv"
dataframe = pd.read_csv(file_path, na_values='?')
#remove as linhas vazias
dataframe = dataframe.dropna()

#retira a coluna Id se caso ela exista
if 'Id' in dataframe.columns:
    dataframe = dataframe.drop('Id', axis=1)

X = dataframe.drop('Class', axis=1).values
y= dataframe['Class'].values

 #aqui acontece a divisão de treino e teste, onde foi separado em 70% dos dados para treino e 30% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42)

#uso da classe
expo_nb = Exponencial_NaiveBayes()
#e de seus metodos
expo_nb.fit(X_treino, y_treino)
y_previsao_exp = expo_nb.previsao(X_teste)

print('--------------------------------------------------------------------------------------')
print('Acuracia com o uso da classe criada Exponencial_NaiveBayes:', accuracy_score(y_teste, y_previsao_exp))
print('--------------------------------------------------------------------------------------')

--------------------------------------------------------------------------------------
Acuracia com o uso da classe criada Exponencial_NaiveBayes: 0.975609756097561
--------------------------------------------------------------------------------------
